# Week 3 Solution: Synthetic Dataset Generator

**Goal:** Build a model that generates synthetic datasets for any project (e.g. **employee records as documents**), with a **Gradio UI** and **JSON** output.

**Scope:** No quantization; we'll use an API (e.g. OpenRouter) to call an LLM for generation.

In [ ]:
# imports

import os
import json
import pandas as pd
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
# Add your config
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

base_url = "https://openrouter.ai/api/v1"
client = OpenAI(base_url=base_url, api_key=openai_api_key)
openai_model = "gpt-4o-mini"
llama_model = "llama3.2"
gemini_model = "gemini-1.5-flash"

system_prompt = """You are a synthetic dataset generator.

Your ONLY task is to output a valid JSON array.

STRICT RULES:
1. Output ONLY a valid JSON array. No markdown, no code fences, no explanation, no text before or after the JSON.
2. The response MUST begin with [ and end with ].
3. Each array element MUST be a single JSON object.
4. Use EXACTLY the keys and structure specified in the user request.
5. Do NOT add extra keys. Do NOT omit required keys.
6. Ensure all values match realistic data types (e.g., strings, numbers, booleans, ISO dates).
7. Ensure the JSON is syntactically valid (no trailing commas, comments, or invalid characters).
8. Generate realistic, varied, and plausible synthetic data.
9. If the user request is ambiguous, assume the simplest valid structure and do not invent extra fields.
10. Do not include duplicate records unless explicitly requested.
11. Ensure IDs are unique within the array.
12. Ensure date formats follow ISO 8601 (YYYY-MM-DD) unless otherwise specified.
"""
# Models
models = [openai_model, llama_model, coder_model]
DATASET_TYPES = ["Employee Records", "Q&A", "Classification", 'Instruction-Response']

### Step 3: Prompt builder

- Write a function that builds the **user prompt** from: topic/domain (e.g. "employee records"), **dataset type**, and **number of records**
- For **Employee Records**, the prompt should ask for a JSON array of objects with fields appropriate for a document (e.g. `name`, `employee_id`, `department`, `role`, `hire_date`, or similar). Decide on a consistent schema so the output is valid JSON

In [ ]:
# Step 3: Implement build_prompt(topic, dataset_type, n) -> user prompt string

def build_prompt(topic: str, dataset_type: str, n: int) -> str:
    n = max(1, min(n, 50))

    if dataset_type == "Employee Records":
        schema = (
            "Each object MUST have exactly these keys: "
            "name (string), employee_id (string, unique), department (string), "
            "role (string), hire_date (string, ISO 8601 YYYY-MM-DD), email (string)."
        )
        return (
            f"Generate exactly {n} synthetic {topic} records as a JSON array. "
            f"{schema} "
            f"Use the domain/topic '{topic}' to make departments, roles, and data realistic. "
            f"Output ONLY the JSON array, no other text."
        )
    elif dataset_type == "Q&A":
        schema = (
            "Each object MUST have: question (string), answer (string), category (string)."
        )
        return (
            f"Generate exactly {n} synthetic Q&A pairs about '{topic}' as a JSON array. "
            f"{schema} "
            f"Output ONLY the JSON array, no other text."
        )
    elif dataset_type == "Classification":
        schema = (
            "Each object MUST have: text (string), label (string), id (string, unique)."
        )
        return (
            f"Generate exactly {n} synthetic classification examples for topic '{topic}' as a JSON array. "
            f"{schema} "
            f"Output ONLY the JSON array, no other text."
        )
    elif dataset_type == "Instruction-Response":
        schema = (
            "Each object MUST have: instruction (string), response (string), id (string, unique)."
        )
        return (
            f"Generate exactly {n} synthetic instruction-response pairs about '{topic}' as a JSON array. "
            f"{schema} "
            f"Output ONLY the JSON array, no other text."
        )
    else:
        return (
            f"Generate exactly {n} synthetic records for topic '{topic}' as a JSON array. "
            f"Each object should have id (string, unique) and other fields appropriate for '{topic}'. "
            f"Output ONLY the JSON array, no other text."
        )

### Step 4: Call the LLM and get raw response

- Create an **OpenAI client** pointing at OpenRouter (same `base_url` + `api_key` as in the reference notebook)
- Implement `generate_dataset(topic, dataset_type, n, model)` that:
  - Sends **system** + **user** message (use your prompt builder)
  - Returns the **raw text** from `response.choices[0].message.content`

In [ ]:
# Step 4: Implement generate_dataset(topic, dataset_type, n, model) -> raw string

def generate_dataset(topic: str, dataset_type: str, n: int, model: str) -> str:
    user_prompt = build_prompt(topic, dataset_type, n)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content or ""

### Step 5: Parse response and save as JSON

- Strip markdown code fences (e.g. \`\`\`json ... \`\`\`) if present
- Parse the string with `json.loads()` to get a list of records
- Optionally convert to a **pandas DataFrame** for preview
- Save the list (or DataFrame) to a **JSON file** (e.g. `synthetic_dataset.json`) with a clear structure (e.g. `orient="records"` for a list of objects)
- Return or expose the file path so Gradio can offer it for download

In [ ]:
# Step 5: Parse raw response, optionally to DataFrame, save as JSON, return path

DEFAULT_OUTPUT_PATH = "dataset.json"

def parse_and_save(raw: str, filepath: str = DEFAULT_OUTPUT_PATH):
    text = raw.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines)
    data = json.loads(text)
    if not isinstance(data, list):
        data = [data]
    df = pd.DataFrame(data)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return filepath, df

### Step 6: Gradio UI

- Create a **Gradio Blocks** interface with:
  - **Inputs:** Topic (text), Dataset type (dropdown from your config), Number of records (slider), Model (dropdown), and optionally export format (we want **JSON** as the main output)
  - **Outputs:** Status message, **preview** (Dataframe or JSON display), and **File** component for **downloading the JSON dataset**
- Wire a **Button** to a function that runs Steps 4–5 and returns (preview data, file path, status message)

In [ ]:
# Step 6: Generate synthetic data

def run_pipeline(topic: str, dataset_type: str, n: float, model: str):
    n = int(n)
    if not topic or not topic.strip():
        return "Enter a topic.", None, None
    try:
        raw_data = generate_dataset(topic.strip(), dataset_type, n, model)
        filepath, df = parse_and_save(raw_data)
        return f"Generated {len(df)} records. Download below.", df, filepath
    except json.JSONDecodeError as e:
        return f"JSON parse error: {e}. Raw response may not be valid JSON.", None, None
    except Exception as e:
        return f"Error: {e}", None, None

In [ ]:
# Step 7:  Build Gradio UI (inputs, outputs, btn.click

with gr.Blocks(title="Synthetic Dataset Generator") as demo:
    gr.Markdown("## Generate synthetic datasets as JSON")
    with gr.Row():
        topic = gr.Textbox(label="Topic", placeholder="e.g. employee records, retail products")
        dataset_type = gr.Dropdown(choices=DATASET_TYPES, value=DATASET_TYPES[0], label="Dataset type")
    with gr.Row():
        n = gr.Slider(1, 100, value=5, step=1, label="Number of records")
        model = gr.Dropdown(choices=models, value=models[0], label="Model")
    btn = gr.Button("Generate dataset")
    status = gr.Textbox(label="Status", interactive=False)
    preview = gr.Dataframe(label="Preview")
    file_out = gr.File(label="Download JSON")
    btn.click(
        fn=run_pipeline,
        inputs=[topic, dataset_type, n, model],
        outputs=[status, preview, file_out],
    )

### Step 8: Launch the app

- Call `demo.launch(share=False)` (or `share=True` for a public link)
- Test with a topic like "employee records" and confirm the downloaded file is valid JSON

In [ ]:
# Step 7: demo.launch(share=False)
demo.launch(share=False)